# New Model: Position-Adjusted Candidate Quality

This notebook builds a baseline click prediction model for Qilin recommendation candidates and uses cross-fitting to estimate position-adjusted candidate quality.

## Modeling Goal

The model predicts whether a displayed recommendation candidate receives a click. After predicting expected click probability from observable features, the notebook compares actual clicks with expected clicks by within-block position.

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = Path("data/qilin_note_metadata_final.parquet")
REPORT_DIR = Path("reports")
MODEL_DIR = Path("models")
FIGURE_DIR = REPORT_DIR / "figures"
REPORT_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

MAX_TRAIN_REQUESTS = 5000
MAX_TEST_REQUESTS = 2000

## Helper Functions

In [ ]:
def load_note_metadata(path=DATA_PATH):
    notes = pd.read_parquet(path).copy()
    impressions = pd.to_numeric(notes["imp_rec_num"], errors="coerce")
    clicks = pd.to_numeric(notes["click_rec_num"], errors="coerce")
    notes["prior_rec_ctr"] = clicks.div(impressions.where(impressions > 0)).clip(0, 1)
    notes["is_commercial"] = (notes["commercial_flag"].fillna(0) != 0).astype(int)
    notes["has_video"] = (notes["video_duration"].fillna(0) > 0).astype(int)
    notes["has_images"] = (notes["image_num"].fillna(0) > 0).astype(int)
    return notes


def flatten_recommendation_requests(dataset, max_requests=None):
    rows = []
    for request_number, request in enumerate(dataset):
        if max_requests is not None and request_number >= max_requests:
            break

        details = request.get("rec_result_details_with_idx") or []
        recent_clicked = request.get("recent_clicked_note_idxs") or []
        query = request.get("query") or ""

        for item in details:
            rows.append(
                {
                    "request_idx": request.get("request_idx"),
                    "session_idx": request.get("session_idx"),
                    "user_idx": request.get("user_idx"),
                    "query": query,
                    "query_length": len(query),
                    "recent_click_count": len(recent_clicked),
                    "note_idx": item.get("note_idx"),
                    "position": item.get("position"),
                    "clicked": int((item.get("click") or 0) > 0),
                    "liked": int((item.get("like") or 0) > 0),
                    "collected": int((item.get("collect") or 0) > 0),
                    "commented": int((item.get("comment") or 0) > 0),
                    "shared": int((item.get("share") or 0) > 0),
                    "page_time": item.get("page_time"),
                }
            )
    return pd.DataFrame(rows)


def add_position_features(frame):
    frame = frame.copy()
    frame["block"] = ((frame["position"] - 1) // 7 + 1).astype(int)
    frame["within_block_position"] = ((frame["position"] - 1) % 7 + 1).astype(int)
    frame["is_sixth_slot"] = (frame["within_block_position"] == 6).astype(int)
    return frame


def build_modeling_table(config_name, max_requests=None, cache_dir="hf_cache"):
    dataset = load_dataset(
        "THUIR/Qilin",
        config_name,
        split="train",
        cache_dir=cache_dir,
    )
    candidates = flatten_recommendation_requests(dataset, max_requests=max_requests)
    notes = load_note_metadata()
    keep_columns = [
        "note_idx",
        "commercial_flag",
        "is_commercial",
        "note_type",
        "content_length",
        "image_num",
        "video_duration",
        "taxonomy1_id",
        "imp_rec_num",
        "click_rec_num",
        "prior_rec_ctr",
        "has_video",
        "has_images",
    ]
    frame = candidates.merge(notes[keep_columns], on="note_idx", how="left")
    frame["taxonomy1_id"] = frame["taxonomy1_id"].fillna("unknown")
    return add_position_features(frame)

## Load Train and Test Data

In [ ]:
train_df = build_modeling_table("recommendation_train", max_requests=MAX_TRAIN_REQUESTS)
test_df = build_modeling_table("recommendation_test", max_requests=MAX_TEST_REQUESTS)

train_df.shape, test_df.shape, train_df.head()

## Model 1-Model 5 Logistic Regression Comparison

These five models match the layered logic in the presentation. Each model adds another group of controls and checks whether the sixth-slot coefficient moves toward zero.

In [ ]:
TARGET = "clicked"

MODEL_SPECS = [
    {
        "model": "Model 1",
        "description": "Position structure only",
        "numeric": [],
        "categorical": ["block", "within_block_position"],
    },
    {
        "model": "Model 2",
        "description": "+ commercial exposure",
        "numeric": ["is_commercial", "commercial_flag"],
        "categorical": ["block", "within_block_position"],
    },
    {
        "model": "Model 3",
        "description": "+ content type and format",
        "numeric": [
            "is_commercial",
            "commercial_flag",
            "note_type",
            "content_length",
            "image_num",
            "video_duration",
            "has_video",
            "has_images",
        ],
        "categorical": ["block", "within_block_position"],
    },
    {
        "model": "Model 4",
        "description": "+ topic category",
        "numeric": [
            "is_commercial",
            "commercial_flag",
            "note_type",
            "content_length",
            "image_num",
            "video_duration",
            "has_video",
            "has_images",
        ],
        "categorical": ["block", "within_block_position", "taxonomy1_id"],
    },
    {
        "model": "Model 5",
        "description": "+ user context and popularity signals",
        "numeric": [
            "query_length",
            "recent_click_count",
            "is_commercial",
            "commercial_flag",
            "note_type",
            "content_length",
            "image_num",
            "video_duration",
            "has_video",
            "has_images",
            "imp_rec_num",
            "click_rec_num",
            "prior_rec_ctr",
        ],
        "categorical": ["block", "within_block_position", "taxonomy1_id"],
    },
]


def feature_list(spec):
    return spec["numeric"] + spec["categorical"]


def make_pipeline(numeric_features, categorical_features):
    transformers = []
    if numeric_features:
        numeric_pipeline = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]
        )
        transformers.append(("numeric", numeric_pipeline, numeric_features))

    if categorical_features:
        categorical_pipeline = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        drop="first",
                        min_frequency=20,
                    ),
                ),
            ]
        )
        transformers.append(("categorical", categorical_pipeline, categorical_features))

    preprocessor = ColumnTransformer(transformers=transformers)
    classifier = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        solver="liblinear",
        random_state=42,
    )
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", classifier),
        ]
    )


def predict_click_probabilities(model, frame, features):
    transformed = model.named_steps["preprocessor"].transform(frame[features])
    if hasattr(transformed, "toarray"):
        transformed = transformed.toarray()
    classifier = model.named_steps["classifier"]
    logits = np.einsum("ij,j->i", transformed, classifier.coef_.ravel()) + classifier.intercept_[0]
    logits = np.clip(logits, -500, 500)
    return 1 / (1 + np.exp(-logits))


def safe_roc_auc(labels, scores):
    if pd.Series(labels).nunique() < 2:
        return np.nan
    return roc_auc_score(labels, scores)


def extract_sixth_slot_effect(model):
    feature_names = model.named_steps["preprocessor"].get_feature_names_out()
    coefs = model.named_steps["classifier"].coef_.ravel()
    coef_table = pd.DataFrame({"feature": feature_names, "coefficient": coefs})
    sixth_slot_rows = coef_table[
        coef_table["feature"].str.contains("within_block_position_6", regex=False)
    ]
    if sixth_slot_rows.empty:
        return np.nan, np.nan
    beta = float(sixth_slot_rows["coefficient"].iloc[0])
    return beta, float(np.exp(beta))

In [ ]:
fitted_models = {}
comparison_rows = []
test_df = test_df.copy()

for spec in MODEL_SPECS:
    features = feature_list(spec)
    fitted_model = make_pipeline(spec["numeric"], spec["categorical"])
    fitted_model.fit(train_df[features], train_df[TARGET])

    score_col = spec["model"].lower().replace(" ", "_") + "_expected_click_probability"
    test_scores = predict_click_probabilities(fitted_model, test_df, features)
    test_df[score_col] = test_scores
    sixth_beta, sixth_odds_ratio = extract_sixth_slot_effect(fitted_model)

    model_metrics = {
        "model": spec["model"],
        "description": spec["description"],
        "num_features_before_encoding": len(features),
        "sixth_slot_beta": sixth_beta,
        "sixth_slot_odds_ratio": sixth_odds_ratio,
        "roc_auc": safe_roc_auc(test_df[TARGET], test_scores),
        "average_precision": average_precision_score(test_df[TARGET], test_scores),
        "train_rows": len(train_df),
        "test_rows": len(test_df),
    }
    comparison_rows.append(model_metrics)
    fitted_models[spec["model"]] = {
        "model": fitted_model,
        "features": features,
        "numeric_features": spec["numeric"],
        "categorical_features": spec["categorical"],
        "score_col": score_col,
        "metrics": model_metrics,
    }

model_comparison = pd.DataFrame(comparison_rows)
model_comparison.to_csv(REPORT_DIR / "model1_model5_logistic_regression.csv", index=False)
model_comparison

## Model 5 Predictions

The rest of the notebook uses `Model 5`, the most complete specification, to estimate expected click probability and adjusted candidate quality.

In [ ]:
model = fitted_models["Model 5"]["model"]
NUMERIC_FEATURES = fitted_models["Model 5"]["numeric_features"]
CATEGORICAL_FEATURES = fitted_models["Model 5"]["categorical_features"]
FEATURES = fitted_models["Model 5"]["features"]

test_df["expected_click_probability"] = test_df[fitted_models["Model 5"]["score_col"]]
test_df["adjusted_performance"] = test_df[TARGET] - test_df["expected_click_probability"]

metrics = fitted_models["Model 5"]["metrics"].copy()
metrics["train_click_rate"] = train_df[TARGET].mean()
metrics["test_click_rate"] = test_df[TARGET].mean()
metrics

## Ranking Metric

In [ ]:
def ndcg_at_k(labels, scores, k=10):
    labels = np.asarray(labels)
    scores = np.asarray(scores)
    if len(labels) == 0:
        return 0.0
    order = np.argsort(scores)[::-1][:k]
    ranked_labels = labels[order]
    gains = (2**ranked_labels - 1) / np.log2(np.arange(2, len(ranked_labels) + 2))
    dcg = gains.sum()
    ideal_order = np.argsort(labels)[::-1][:k]
    ideal_labels = labels[ideal_order]
    ideal_gains = (2**ideal_labels - 1) / np.log2(np.arange(2, len(ideal_labels) + 2))
    ideal_dcg = ideal_gains.sum()
    return dcg / ideal_dcg if ideal_dcg > 0 else 0.0


def mean_ndcg_at_k(frame, label_col, score_col, group_col="request_idx", k=10):
    values = []
    for _, group in frame.groupby(group_col):
        values.append(ndcg_at_k(group[label_col], group[score_col], k=k))
    return float(np.mean(values)) if values else 0.0


metrics["mean_ndcg_at_10"] = mean_ndcg_at_k(
    test_df,
    label_col=TARGET,
    score_col="expected_click_probability",
    k=10,
)
metrics

## Cross-Fitted Candidate Quality

In [ ]:
def add_cross_fitted_predictions(frame, n_splits=5):
    frame = frame.copy().reset_index(drop=True)
    unique_groups = frame["request_idx"].nunique()
    n_splits = min(n_splits, unique_groups)
    splitter = GroupKFold(n_splits=n_splits)
    out_of_fold_scores = np.zeros(len(frame))

    for fold, (train_idx, valid_idx) in enumerate(
        splitter.split(frame, frame[TARGET], groups=frame["request_idx"]),
        start=1,
    ):
        fold_model = make_pipeline(NUMERIC_FEATURES, CATEGORICAL_FEATURES)
        fold_model.fit(frame.loc[train_idx, FEATURES], frame.loc[train_idx, TARGET])
        out_of_fold_scores[valid_idx] = predict_click_probabilities(
            fold_model,
            frame.loc[valid_idx],
            FEATURES,
        )
        print(f"Finished fold {fold}/{n_splits}")

    frame["expected_click_probability"] = out_of_fold_scores
    frame["adjusted_performance"] = frame[TARGET] - frame["expected_click_probability"]
    return frame


cross_fit_df = add_cross_fitted_predictions(train_df, n_splits=5)
cross_fit_df[[TARGET, "expected_click_probability", "adjusted_performance"]].head()

In [ ]:
slot_quality = (
    cross_fit_df.groupby("within_block_position")
    .agg(
        n=("clicked", "size"),
        click_rate=("clicked", "mean"),
        expected_click_rate=("expected_click_probability", "mean"),
        avg_adjusted_quality=("adjusted_performance", "mean"),
        commercial_rate=("is_commercial", "mean"),
    )
    .reset_index()
)
slot_quality.to_csv(REPORT_DIR / "cross_fitted_slot_quality.csv", index=False)
slot_quality

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#d62728" if slot == 6 else "#4c78a8" for slot in slot_quality["within_block_position"]]
ax.bar(slot_quality["within_block_position"], slot_quality["avg_adjusted_quality"], color=colors)
ax.axhline(0, color="black", linewidth=1)
ax.set_xlabel("Within-block position")
ax.set_ylabel("Average adjusted performance")
ax.set_title("Cross-Fitted Candidate Quality by Within-Block Position")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "cross_fitted_candidate_quality.png", dpi=160)
plt.show()

## Save Model and Metrics

In [ ]:
metrics_path = REPORT_DIR / "new_model_metrics.csv"
pd.Series(metrics, name="value").to_csv(metrics_path)
joblib.dump(model, MODEL_DIR / "new_model.joblib")

metrics_path, MODEL_DIR / "new_model.joblib"

## Takeaway

The model estimates each candidate's expected click probability from observable position, note, topic, and context features. The adjusted-performance comparison then checks whether sixth-slot candidates underperform after accounting for those observable factors.